In [556]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dtale
from pandas.io.xml import preprocess_data
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder,Binarizer, TargetEncoder

In [557]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

In [558]:
# d = dtale.show(train_data)
# d.open_browser()

Проводим исследование, осматриваем данные, убираем лишние столбцы

In [559]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [560]:
numeric_features = ['LotFrontage','LotArea','MasVnrArea','TotalBsmtSF','GrLivArea','GarageYrBlt','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','ScreenPorch','PoolArea','MiscVal','YearBuilt','1stFlrSF','2ndFlrSF','Fireplaces','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd']
nominal_features = ['MSSubClass','MSZoning','LotShape','LotConfig','Condition1','BldgType','HouseStyle','RoofStyle','RoofMatl','MasVnrType','Heating','Electrical','Neighborhood','Functional','GarageType','GarageFinish','PavedDrive','SaleType','SaleCondition','Exterior1st','Street','Alley','CentralAir']
ordinal_features = ['LandContour','Utilities','LandSlope','OverallQual','OverallCond','BsmtQual','BsmtExposure','HeatingQC','KitchenQual','GarageQual','ExterQual','ExterCond']

In [561]:
# Удаляем дубликаты и не нужные столбцы
train_data.drop_duplicates(inplace=True)
test_data.drop_duplicates(inplace=True)

test_ids = test_data['Id']
train_data = train_data.drop(['Id','Fence','MiscFeature'], axis=1)
test_data= test_data.drop(['Id','Fence','MiscFeature'], axis=1)

In [562]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 78 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   str    
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   str    
 5   Alley          91 non-null     str    
 6   LotShape       1460 non-null   str    
 7   LandContour    1460 non-null   str    
 8   Utilities      1460 non-null   str    
 9   LotConfig      1460 non-null   str    
 10  LandSlope      1460 non-null   str    
 11  Neighborhood   1460 non-null   str    
 12  Condition1     1460 non-null   str    
 13  Condition2     1460 non-null   str    
 14  BldgType       1460 non-null   str    
 15  HouseStyle     1460 non-null   str    
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt      1460

In [563]:
test_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 77 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1459 non-null   int64  
 1   MSZoning       1455 non-null   str    
 2   LotFrontage    1232 non-null   float64
 3   LotArea        1459 non-null   int64  
 4   Street         1459 non-null   str    
 5   Alley          107 non-null    str    
 6   LotShape       1459 non-null   str    
 7   LandContour    1459 non-null   str    
 8   Utilities      1457 non-null   str    
 9   LotConfig      1459 non-null   str    
 10  LandSlope      1459 non-null   str    
 11  Neighborhood   1459 non-null   str    
 12  Condition1     1459 non-null   str    
 13  Condition2     1459 non-null   str    
 14  BldgType       1459 non-null   str    
 15  HouseStyle     1459 non-null   str    
 16  OverallQual    1459 non-null   int64  
 17  OverallCond    1459 non-null   int64  
 18  YearBuilt      1459

In [564]:
X = train_data.drop('SalePrice',axis=1)
y = np.log1p(train_data['SalePrice'])

In [565]:
preprocessor = ColumnTransformer(
    transformers=[
        # Numeric медиана + scaling
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]),numeric_features),

        # Nominal - OHE + обработка пропусков
        ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False),nominal_features),

        # Ordinal : мода + OE
        ('ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)),
        ]),ordinal_features),

        # Кодирование очень большого признака
        ('target_enc', TargetEncoder(), ['Neighborhood']),
    ],
    remainder='drop'
)

In [566]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureCreator(BaseEstimator,TransformerMixin):
    def __init__(self):
        self.bad_condition = ['Artery','Feedr','RRNn','RRAn','RRNe','RRAe']
        self.quality_materials = {
            'Stone': 3, 'BrkFace': 2, 'CemntBd': 1, 'VinylSd': 1
        }

    def fit(self,X, y=None):
        if y is not None:
             self.neigh_means_ = pd.Series(y).groupby(X['Neighborhood']).mean()
        return self

    def transform(self,X):
        X_new = X.copy()
        # Новые признаки:

        # Выявление близкости к дороге и шуму
        X_new['HasBadCondition'] = (
            X_new['Condition1'].fillna('None').isin(self.bad_condition) |
            X_new['Condition2'].fillna('None').isin(self.bad_condition)
        ).astype(int)
        X_new = X_new.drop('Condition2', axis=1)

        # Вычисление возраста дома
        X_new['HouseAge'] = X_new['YrSold'] - X_new['YearBuilt']
        # Наличие ремонта в доме
        X_new['RemodAge'] = X_new['YrSold'] - X_new['YearRemodAdd']

        # Вычисление дорого материала для фасада крыши
        ext1 = X_new['Exterior1st'].fillna('None').map(self.quality_materials).fillna(0)
        ext2 = X_new['Exterior2nd'].fillna('None').map(self.quality_materials).fillna(0)
        X_new['GoodExterior'] = np.maximum(ext1, ext2)

        # Вычисление всей площади дома + подвал
        X_new['TotalArea'] = (X_new['TotalBsmtSF'].fillna(0) + X_new['1stFlrSF'].fillna(0) + X_new['2ndFlrSF'].fillna(0))

        # Количетсво санузлов в доме
        X_new['TotalBaths'] = (
            X_new['BsmtFullBath'].fillna(0) +
            X_new['FullBath'].fillna(0) +
            (
                    0.5 * X_new['BsmtHalfBath'].fillna(0) + X_new['HalfBath'].fillna(0)
             )
        )
        # Качество × площадь
        X_new['QualitySF'] = X_new['OverallQual'] * X_new['GrLivArea']
        X_new['OverallScore'] = X_new['OverallQual'] + X_new['OverallCond']

        # Есть ли подвал / гараж
        X_new['HasBsmt'] = (X_new['TotalBsmtSF'] > 0).astype(int)
        X_new['HasGarage'] = (X_new['GarageArea'] > 0).astype(int)

        # Kitchen + Quality
        X_new['KitchenScore'] = X_new['KitchenAbvGr'] * X_new['OverallQual']

        # обработка Neighborhood
        neigh_price = X_new['Neighborhood'].map(self.neigh_means_).fillna(180000)
        X_new['NeighborhoodPrice'] = neigh_price

        return X_new

In [567]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

pipeline = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

# Тестирование
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE на тесте: {rmse_test:.4f}")


RMSE CV: 0.1524
RMSE на тесте: 0.1324


In [568]:
from sklearn.ensemble import RandomForestRegressor

pipeline02 = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

pipeline02.fit(X_train, y_train)
y_pred_02 = pipeline02.predict(X_test)

scores = cross_val_score(pipeline02, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

rmse_test02 = np.sqrt(mean_squared_error(y_test, y_pred_02))
print(f"RMSE на тесте: {rmse_test02:.4f}")


RMSE CV: 0.1463
RMSE на тесте: 0.1468


In [569]:
from sklearn.ensemble import GradientBoostingRegressor

pipeline03 = Pipeline([
    ('features', FeatureCreator()),
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42))
])

pipeline03.fit(X_train, y_train)
y_pred_03 = pipeline03.predict(X_test)

scores = cross_val_score(pipeline03, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
rmse = np.sqrt(-scores.mean())
print(f"RMSE CV: {rmse:.4f}")

rmse_test03 = np.sqrt(mean_squared_error(y_test, y_pred_03))
print(f"RMSE на тесте: {rmse_test03:.4f}")


RMSE CV: 0.1323
RMSE на тесте: 0.1382


In [570]:
pipeline03.fit(X, y)
y_test_pred = pipeline03.predict(test_data)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': np.expm1(y_test_pred)
})

submission.to_csv('submission_new_features_02.csv', index=False)